In [ ]:
import os
import json
import time
import random
import argparse
from datetime import datetime

import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split

from torchvision import datasets, transforms

from tqdm import tqdm

import matplotlib.pyplot as plt

#для воиспроизводимости результатов
def set_seed(seed: int = 42, deterministic: bool = False):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


In [2]:
device = 'mps' if torch.backends.mps.is_available() else 'cpu'

### Tensorboard

In [3]:
class MLP(nn.Module):
    def __init__(self, in_features=28 * 28, hidden1=256, num_classes=10):
        super().__init__()
        
        #с помощью nn.sequential можно создавать блоки слоев
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden1),
            nn.ReLU(inplace=True),
            nn.Linear(hidden1, num_classes),
        )

    def forward(self, x):
        x = x.view(x.size(0), -1) #вытягиваем картинку в одномерный массив
        x = self.net(x)
        
        return x


In [4]:
#делаем confusion matrix 
def plot_confusion_matrix(cm, class_names):

    fig, ax = plt.subplots(figsize=(5, 5), dpi=120)
    im = ax.imshow(cm, cmap="Blues")
    ax.figure.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    ax.set_xticks(np.arange(len(class_names)))
    ax.set_yticks(np.arange(len(class_names)))
    ax.set_xticklabels(class_names)
    ax.set_yticklabels(class_names)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

    thresh = cm.max() / 2.0 if cm.size else 0.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(
                j, i, format(cm[i, j], "d"),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black"
            )
    fig.tight_layout()
    return fig


def make_misclassified_figure(images, y_true, y_pred, max_items=16):
    """
    images: list of tensors or np arrays (N, 1, 28, 28)
    y_true, y_pred: lists/arrays
    """
    n = min(len(images), max_items)
    cols = 4
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.2, rows * 2.2), dpi=120)
    if rows == 1 and cols == 1:
        axes = np.array([[axes]])
    elif rows == 1:
        axes = np.array([axes])
    elif cols == 1:
        axes = np.array([[ax] for ax in axes])

    for i in range(rows * cols):
        r, c = divmod(i, cols)
        ax = axes[r, c]
        ax.axis("off")
        if i < n:
            img = images[i]
            if hasattr(img, "detach"):
                img = img.detach().cpu().numpy()
            img = np.squeeze(img)
            ax.imshow(img, cmap="gray")
            ax.set_title(f"T:{y_true[i]} P:{y_pred[i]}", fontsize=9)
    fig.tight_layout()
    return fig


In [5]:
train_full = datasets.MNIST(
    root="./data", train=True, transform=transforms.ToTensor(), download=True
)
#аргумент transforms получает на вход некоторый набор функций, которые будут применяться к данным внутри датасета

test_set = datasets.MNIST(
    root="./data", train=False, transform=transforms.ToTensor(), download=True
)

#делим на train/val/test 
val_size = int(len(train_full) * 0.1)
train_size = len(train_full) - val_size

train_set, val_set = random_split(train_full, [train_size, val_size])


BATCH_SIZE = 128

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)

100%|██████████| 9.91M/9.91M [00:10<00:00, 906kB/s] 
100%|██████████| 28.9k/28.9k [00:00<00:00, 190kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 1.75MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 4.56MB/s]


In [ ]:
HIDDEN_SZ = 128
LR = 0.01

# Модель, лосс, оптимайзер
model = MLP(hidden1=HIDDEN_SZ).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

In [7]:
#отключение градиентов для валидации/тестирования можно сделать также через декоратор
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total = 0

    all_preds = []
    all_targets = []
    for images, targets in loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        outputs = model(images)
        loss = criterion(outputs, targets)

        total_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        total_correct += (preds == targets).sum().item()
        total += images.size(0)

        all_preds.append(preds.cpu())
        all_targets.append(targets.cpu())

    all_preds = torch.cat(all_preds).numpy()
    all_targets = torch.cat(all_targets).numpy()
    avg_loss = total_loss / max(total, 1)
    acc = float(total_correct) / float(max(total, 1))
    
    return avg_loss, acc, all_targets, all_preds

In [8]:
from torch.utils.tensorboard import SummaryWriter

In [9]:
# Подготовим tensorboard 
run_name = f"mlp_h1{HIDDEN_SZ}_lr{LR}_{datetime.now().strftime('%Y%m%d-%H%M%S')}"
run_dir = os.path.join("runs/mnist_fc", run_name)
os.makedirs(run_dir, exist_ok=True)

writer = SummaryWriter(log_dir=run_dir)

### Добавим граф модели

In [10]:
dummy = torch.randn(1, 1, 28, 28, device=device)
writer.add_graph(model, dummy)

In [ ]:
best_val_acc = 0.0
global_step = 0
EPOCHS = 25


for epoch in tqdm(range(1, EPOCHS + 1)):
    model.train()
    running_loss = 0.0
    running_correct = 0
    running_total = 0
    epoch_start = time.time()

    for images, targets in train_loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            preds = outputs.argmax(dim=1)
            running_correct += (preds == targets).sum().item()
            running_total += images.size(0)
            running_loss += loss.item() * images.size(0)

        global_step += 1

    train_loss = running_loss / max(running_total, 1)
    train_acc = float(running_correct) / float(max(running_total, 1))

    # Валидация
    val_loss, val_acc, y_true_val, y_pred_val = evaluate(model, val_loader, criterion, device)

    # Тест + визуализации (матрица ошибок, примеры ошибок)
    test_loss, test_acc, y_true_test, y_pred_test = evaluate(model, test_loader, criterion, device)
   
    #cm_test = np.zeros((10, 10), dtype=np.int64)
    #np.add.at(cm_test, (y_true_test, y_pred_test), 1)
    #fig_cm = plot_confusion_matrix(cm_test, [str(i) for i in range(10)])

    # Логи в TensorBoard
    writer.add_scalar("train/loss", train_loss, epoch)
    writer.add_scalar("train/accuracy", train_acc, epoch)
    writer.add_scalar("val/loss", val_loss, epoch)
    writer.add_scalar("val/accuracy", val_acc, epoch)
    writer.add_scalar("test/loss", test_loss, epoch)
    writer.add_scalar("test/accuracy", test_acc, epoch)

    # Гистограммы весов слоёв
    for name, param in model.named_parameters():
        writer.add_histogram(f"params/{name}", param.detach().cpu().numpy(), epoch)

    # Изображения: матрица ошибок и примеры ошибок
    #writer.add_figure("images/confusion_matrix_test", fig_cm, global_step=epoch, close=True)

  0%|          | 0/25 [00:00<?, ?it/s]/Users/timmiakov/Dev/hse-ml-course/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
100%|██████████| 25/25 [01:02<00:00,  2.50s/it]
